# Synthetic SFT Data Generator

Uses OpenRouter (qwen/qwen3.5-flash-02-23) to generate ~3000 synthetic SFT training records
from the existing 260 real keywords, expanded by category-specific keyword variants.

**Output**:
- `data/sft/keyword-scraper-2/synthetic_justifier_sft.jsonl`
- `data/sft/keyword-scraper-2/synthetic_enricher_sft.jsonl`

In [1]:
# ─── Setup ────────────────────────────────────────────────────────────────────

import os
import json
import asyncio
import re
import time
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

import sys
sys.path.insert(0, str(Path.cwd().parent))

from services.llm.prompts import (
    JUSTIFIER_SYSTEM, ENRICHER_SYSTEM,
    build_justifier_prompt, build_enricher_prompt,
    build_messages,
)
from shared.shared.constants import KeywordStatus

# ─── Config ──────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
MODEL = "qwen/qwen3.5-flash-02-23"
MAX_CONCURRENT = 5
KEYWORDS_PER_CATEGORY = 26
ARTICLES_PER_KEYWORD = 3

OUTPUT_DIR = Path("data/sft/keyword-scraper-2")
JUSTIFIER_OUT = OUTPUT_DIR / "synthetic_justifier_sft.jsonl"
ENRICHER_OUT = OUTPUT_DIR / "synthetic_enricher_sft.jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE)


def chat(model: str, messages: list[dict], *, max_retries=3) -> str:
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0.7,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = (attempt + 1) * 5 + (attempt * 2)
            print(f"  ⚠ Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"chat() failed after {max_retries} attempts")


CATEGORIES = {
    "integritas_penegakan_hukum": (
        "kasus korupsi, suap, gratifikasi, penegakan hukum oleh KPK, Polri, kejaksaan, "
        "serta isu integritas, transparansi, dan etika pejabat publik"
    ),
    "kebijakan_layanan_publik": (
        "kebijakan pemerintah pusat dan daerah, perpres, permen, undang-undang, "
        "serta efektivitas dan kualitas penyediaan layanan publik bagi masyarakat"
    ),
    "kinerja_pemerintah": (
        "evaluasi kinerja kabinet, kementerian, pemerintah daerah, DPR/DPRD, "
        "serta capaian program, birokrasi, dan tata kelola pemerintahan secara umum"
    ),
    "keamanan_siber_ketertiban_digital": (
        "keamanan siber, perlindungan data pribadi, penanganan peretasan, "
        "literasi digital, ketertiban ruang siber, dan regulasi elektronik"
    ),
    "kesejahteraan_bantuan_sosial": (
        "program perlindungan sosial, pengentasan kemiskinan, subsidi, "
        "bantuan sosial (bansos), PKH, dan jaminan sosial masyarakat"
    ),
    "infrastruktur_layanan_transportasi": (
        "proyek pembangunan fisik seperti jalan tol, jembatan, bandara, pelabuhan, "
        "serta ketersediaan, kualitas, dan tata kelola layanan transportasi publik"
    ),
    "ekonomi_digital": (
        "perkembangan industri teknologi, startup, e-commerce, fintech, "
        "digitalisasi UMKM, dan inovasi perekonomian berbasis digital"
    ),
    "ketenagakerjaan": (
        "ketersediaan lapangan kerja, tingkat pengangguran, isu buruh, "
        "regulasi upah minimum, hak pekerja, dan dinamika hubungan industrial"
    ),
    "layanan_kesehatan": (
        "akses dan kualitas fasilitas kesehatan, BPJS Kesehatan, JKN, "
        "penanganan wabah, gizi buruk, dan kebijakan kesehatan masyarakat"
    ),
    "pendidikan_pengembangan_sdm": (
        "sistem pendidikan dasar hingga tinggi, kurikulum, beasiswa pemerintah, "
        "kesejahteraan guru, dan program peningkatan kapasitas sumber daya manusia"
    ),
    "krisis_sosial_kebencanaan": (
        "mitigasi dan penanganan darurat bencana alam, perubahan iklim, "
        "kerusakan lingkungan, krisis sosial, dan manajemen tanggap darurat"
    ),
}

print(f"✓ Loaded {len(CATEGORIES)} categories")

✓ Loaded 11 categories


In [2]:
# ─── Load real keyword seeds from existing JSONL ─────────────────────────────────
# Load keywords from the already-exported real SFT data — no DB needed.

SEED_FILE = OUTPUT_DIR / "justifier_sft.jsonl"

real_keyword_texts = []
with open(SEED_FILE) as f:
    for line in f:
        record = json.loads(line)
        user_content = next(
            m["content"] for m in record["messages"] if m["role"] == "user"
        )
        # Extract keyword from "Keyword trending: X"
        m = re.search(r"Keyword trending: (.+?)(?:\n|$)", user_content)
        if m:
            real_keyword_texts.append(m.group(1).strip())

print(f"Loaded {len(real_keyword_texts)} real keywords as seeds")
print(f"  Sample: {real_keyword_texts[:5]}")


RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
# ─── Step 1: Generate category-specific keyword variants ───────────────────────

KEYWORD_SYNTHESIS_PROMPT = """Kamu adalah generator kata kunci tren untuk sistem monitoring isu pemerintahan Indonesia.

Untuk kategori: {category_name}
Deskripsi: {category_desc}

Buat tepat {count} kata kunci tren Indonesia yang relevan dengan kategori ini.
Kata kunci harus:
- Berbahasa Indonesia (bisa campur Inggris jika umum)
- Spesifik, bukan generik (misal: "pemadatan kebakaran Jakarta" bukan "berita jakarta")
- Mencerminkan topik yang sedang tren atau penting di Indonesia
- Variatif: campur antara nama lembaga, kebijakan, lokasi, dan isu spesifik

Jika seed diberikan, buat variasi yang TERINSPIRASI dari seed tapi TIDAK SAMA PERSIS:
Seed: {seed}

Respond HANYA dengan JSON array string, tidak ada teks lain.
Format: ["kata kunci 1", "kata kunci 2", ...]
"""


def generate_category_keywords(category_key: str, category_desc: str, count: int, seed: str) -> list[str]:
    prompt = KEYWORD_SYNTHESIS_PROMPT.format(
        category_name=category_key,
        category_desc=category_desc,
        count=count,
        seed=seed,
    )
    messages = [{"role": "user", "content": prompt}]
    try:
        response = chat(MODEL, messages)
        match = re.search(r'\[.*\]', response, re.DOTALL)
        if match:
            keywords = json.loads(match.group())
            return [k.strip() for k in keywords if k.strip()]
        print(f"  ⚠ Could not parse: {response[:100]}")
        return []
    except Exception as e:
        print(f"  ⚠ Error: {e}")
        return []


all_synthetic = []
cat_keys = list(CATEGORIES.keys())

for i, (cat_key, cat_desc) in enumerate(CATEGORIES.items()):
    seed = real_keyword_texts[i % len(real_keyword_texts)]
    half = KEYWORDS_PER_CATEGORY // 2

    from_seed = generate_category_keywords(cat_key, cat_desc, half, seed)
    from_scratch = generate_category_keywords(cat_key, cat_desc, KEYWORDS_PER_CATEGORY - half, "(tidak ada seed)")

    combined = from_seed + from_scratch
    seen = set()
    unique = []
    for k in combined:
        kl = k.lower()
        if kl not in seen:
            seen.add(kl)
            unique.append(k)

    for kw in unique:
        all_synthetic.append({"keyword": kw, "category": cat_key})

    print(f"  [{i+1:02d}/{len(CATEGORIES)}] {cat_key}: {len(unique)} keywords")
    time.sleep(0.5)

print(f"\n✓ Total synthetic keywords: {len(all_synthetic)}")

In [ ]:
# ─── Step 2: Generate synthetic articles per keyword ─────────────────────────

ARTICLE_SYNTHESIS_PROMPT = """Kamu adalah generator artikel sintetis untuk data training model AI.

Buat {count} artikel pendek (seperti artikel berita Indonesia) untuk kata kunci: {keyword}

Setiap artikel HARUS terdiri dari:
- "title": judul artikel (maksimal 12 kata, spesifik)
- "body": isi artikel 2-3 paragraf (150-300 kata per artikel), gaya berita Indonesia
- "summary": ringkasan 1-2 kalimat (maksimal 50 kata)

Artikel harus:
- Berbahasa Indonesia formal (gaya berita)
- Mengandung nama lembaga pemerintah, kebijakan, atau tokoh resmi Indonesia
- Netral dan faktual, bukan opini

Respond HANYA dengan JSON array, tidak ada teks lain.
Format:
[{{"title": "...", "body": "...", "summary": "..."}}, ...]
"""


def parse_articles_response(response: str) -> list[dict]:
    match = re.search(r'\[.*\]', response, re.DOTALL)
    if match:
        try:
            articles = json.loads(match.group())
            if isinstance(articles, list) and len(articles) > 0:
                return articles
        except json.JSONDecodeError:
            pass
    print(f"  ⚠ Parse error: {response[:100]}")
    return []


class SyntheticArticle:
    def __init__(self, title: str, body: str, summary: str):
        self.title = title
        self.body = body
        self.summary = summary


def generate_articles_for_keyword(keyword: str, count: int = ARTICLES_PER_KEYWORD) -> list[SyntheticArticle]:
    prompt = ARTICLE_SYNTHESIS_PROMPT.format(keyword=keyword, count=count)
    messages = [{"role": "user", "content": prompt}]
    try:
        response = chat(MODEL, messages)
        raw = parse_articles_response(response)
        return [
            SyntheticArticle(
                title=a.get("title", ""),
                body=a.get("body", ""),
                summary=a.get("summary", ""),
            )
            for a in raw
            if a.get("title") and a.get("body")
        ]
    except Exception as e:
        print(f"  ⚠ Article error for '{keyword}': {e}")
        return []


print(f"\n🔧 Generating {ARTICLES_PER_KEYWORD} articles per keyword...")
print(f"   Total: {len(all_synthetic)} keywords × {ARTICLES_PER_KEYWORD} = {len(all_synthetic) * ARTICLES_PER_KEYWORD} articles")

for idx, item in enumerate(all_synthetic):
    articles = generate_articles_for_keyword(item["keyword"])
    item["articles"] = articles
    if (idx + 1) % 10 == 0:
        print(f"  [{idx+1}/{len(all_synthetic)}] processed")
    time.sleep(0.3)  # rate limit protection

with_articles = [it for it in all_synthetic if it["articles"]]
print(f"\n✓ Keywords with articles: {len(with_articles)} / {len(all_synthetic)}")

In [ ]:
# ─── Step 3: Run Justifier ─────────────────────────────────────────────────

def build_context(articles: list[SyntheticArticle]) -> str:
    parts = []
    for i, a in enumerate(articles, 1):
        parts.append(f"[Artikel {i}] {a.title}\n{a.body}")
    return "\n\n".join(parts)


def run_justifier(keyword: str, articles: list[SyntheticArticle]) -> dict:
    ctx = build_context(articles)
    user_prompt = build_justifier_prompt(keyword, ctx)
    messages = build_messages(JUSTIFIER_SYSTEM, user_prompt)
    try:
        response = chat(MODEL, messages)
        clean = response.strip()
        if clean.startswith("```json"): clean = clean[7:]
        elif clean.startswith("```"): clean = clean[3:]
        if clean.endswith("```"): clean = clean[:-3]
        data = json.loads(clean.strip())
        return {"is_relevant": bool(data.get("is_relevant")), "justification": str(data.get("justification", ""))}
    except (json.JSONDecodeError, KeyError) as e:
        print(f"  ⚠ Justifier error '{keyword[:30]}...': {e}")
        return {"is_relevant": False, "justification": "Parse error"}


print(f"\n🔧 Running Justifier on {len(with_articles)} keywords...")
for idx, item in enumerate(with_articles):
    result = run_justifier(item["keyword"], item["articles"])
    item["just_result"] = result
    if (idx + 1) % 20 == 0:
        print(f"  [{idx+1}/{len(with_articles)}] processed")
    time.sleep(0.3)

relevant = [it for it in with_articles if it["just_result"]["is_relevant"]]
print(f"\n✓ Justifier done: {len(relevant)}/{len(with_articles)} relevant")

In [ ]:
# ─── Step 4: Run Enricher ─────────────────────────────────────────────────

def run_enricher(keyword: str, articles: list[SyntheticArticle]) -> dict:
    ctx = build_context(articles)
    user_prompt = build_enricher_prompt(keyword, ctx)
    messages = build_messages(ENRICHER_SYSTEM, user_prompt)
    try:
        response = chat(MODEL, messages)
        clean = response.strip()
        if clean.startswith("```json"): clean = clean[7:]
        elif clean.startswith("```"): clean = clean[3:]
        if clean.endswith("```"): clean = clean[:-3]
        data = json.loads(clean.strip())
        kw_list = data.get("expanded_keywords", [])
        return {"expanded_keywords": kw_list if isinstance(kw_list, list) else []}
    except (json.JSONDecodeError, KeyError) as e:
        print(f"  ⚠ Enricher error '{keyword[:30]}...': {e}")
        return {"expanded_keywords": [keyword]}


print(f"\n🔧 Running Enricher on {len(relevant)} relevant keywords...")
for idx, item in enumerate(relevant):
    result = run_enricher(item["keyword"], item["articles"])
    item["enrich_result"] = result
    if (idx + 1) % 20 == 0:
        print(f"  [{idx+1}/{len(relevant)}] processed")
    time.sleep(0.3)

enriched = [it for it in relevant if it.get("enrich_result")]
print(f"\n✓ Enricher done: {len(enriched)} keywords enriched")

In [ ]:
# ─── Step 5: Write JSONL ─────────────────────────────────────────────────

for path in [JUSTIFIER_OUT, ENRICHER_OUT]:
    if path.exists():
        path.unlink()

just_count = 0
enr_count = 0

for item in with_articles:
    kw = item["keyword"]
    arts = item["articles"]
    just = item["just_result"]
    enrich = item.get("enrich_result")
    ctx = build_context(arts)

    # ── Justifier record ──
    user_prompt = build_justifier_prompt(kw, ctx)
    messages = build_messages(JUSTIFIER_SYSTEM, user_prompt)
    assistant_content = json.dumps({
        "is_relevant": just["is_relevant"],
        "justification": just["justification"],
    }, ensure_ascii=False)
    record = {"messages": messages + [{"role": "assistant", "content": assistant_content}]}
    with open(JUSTIFIER_OUT, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    just_count += 1

    # ── Enricher record ──
    if just["is_relevant"] and enrich:
        user_prompt = build_enricher_prompt(kw, ctx)
        messages = build_messages(ENRICHER_SYSTEM, user_prompt)
        assistant_content = json.dumps({
            "expanded_keywords": enrich["expanded_keywords"],
        }, ensure_ascii=False)
        record = {"messages": messages + [{"role": "assistant", "content": assistant_content}]}
        with open(ENRICHER_OUT, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        enr_count += 1

print(f"\n✅ Export complete!")
print(f"   Justifier: {just_count} records → {JUSTIFIER_OUT}")
print(f"   Enricher:  {enr_count} records → {ENRICHER_OUT}")

In [ ]:
# ─── Preview & Stats ───────────────────────────────────────────────────

import subprocess

print("=" * 60)
print("JUSTIFIER SAMPLE:")
print("=" * 60)
with open(JUSTIFIER_OUT) as f:
    d = json.loads(f.readline())
    for m in d["messages"]:
        print(f"\n[{m['role'].upper()}]\n{m['content'][:300]}")

print("\n" + "=" * 60)
print("ENRICHER SAMPLE:")
print("=" * 60)
with open(ENRICHER_OUT) as f:
    d = json.loads(f.readline())
    for m in d["messages"]:
        print(f"\n[{m['role'].upper()}]\n{m['content'][:300]}")

print("\n" + "=" * 60)
print("FILE STATS:")
print("=" * 60)
result = subprocess.run(["wc", "-l", str(JUSTIFIER_OUT), str(ENRICHER_OUT)], capture_output=True, text=True)
print(result.stdout)